# Parameters

In [1]:
from common.utils import (
    check_initial_sector_omega_ratio,
    omega_c,
    delta0_from_N_Gamma,
    Omega0_from_N_Gamma,
    default_three_phase_protocol,
    validated_mcwf_dt,
)

from pathlib import Path
import numpy as np

%load_ext autoreload
%autoreload 2

seed = 1234
num_snapshots = 100

### Simulation

In [ ]:
def compare_homogeneous_vs_inhomogeneous(
    N=100,
    dN=0,
    N1=0,
    dt=1e-2,
    Gamma=1.0,
    ntraj=10,
    shifted_jump_operator=True,
    omega_1=1.0,
    n_processes=1,
):
    import time

    from common.utils import Omega_Gamma_from_cavity_parameters
    from quantum_trajectories.state_helpers import (
        centered_sector_initial_coeffs,
        centered_group_resolved_sector_initial_coeffs,
    )
    from quantum_trajectories.ensamble_sim import run_trajectory_ensemble
    from quantum_trajectories.aggregator import (
        ensemble_observables,
        make_averaged_result,
    )
    from common.plotting import plot_trajectory_angles_and_excitation
    from quantum_trajectories.inhomogeneous_diagnostics import (
        plot_inhomogeneous_group_angles,
        plot_inhomogeneous_mfe_residuals,
    )
    from quantum_trajectories.dephasing_diagnostics import plot_compare_dephasing_bloch_lengths
    from quantum_trajectories.squeezing import (
        plot_generalized_xi,
        plot_inhomogeneous_generalized_xi,
    )

    # Model and parameters
    Omega_ratio = 0.4
    Omega0 = Omega_ratio * omega_c(N_J, Gamma)
    delta0 = 1

    N_J = N // 2
    N2 = N-N1
    
    # protocol phases
    phases = default_three_phase_protocol(
        T1=5.0,
        T2=500.0,
        T3=5.0,
        delta0=delta0,
        Omega0=Omega0,
    )

    # sector coefficients
    # homogeneous_sector_coeffs = centered_sector_initial_coeffs(
    #     N,
    #     dN=dN,
    #     sector_distribution="binomial",
    # )
    inhomogeneous_sector_coeffs = centered_group_resolved_sector_initial_coeffs(
        N,
        dN=dN,
        N1=N1,
        N2=N2,
        sector_distribution="binomial",
    )

    # homogeneous_ratio_check = check_initial_sector_omega_ratio(
    #     homogeneous_sector_coeffs,
    #     Omega=max(abs(phase.omega) for phase in phases),
    #     Gamma=Gamma,
    # )
    # if not homogeneous_ratio_check["is_valid"]:
    #     print(
    #         "Omega/Omega_c check not valid for homogeneous run: "
    #         f"Omega={homogeneous_ratio_check['omega']}, Omega_c={homogeneous_ratio_check['omega_c']}, "
    #         f"smallest Nj={homogeneous_ratio_check['min_nj']}, ratio={homogeneous_ratio_check['ratio']}"
    #     )
    #     return None

    inhomogeneous_ratio_check = check_initial_sector_omega_ratio(
        inhomogeneous_sector_coeffs,
        Omega=max(abs(phase.omega) for phase in phases),
        Gamma=Gamma,
    )
    if not inhomogeneous_ratio_check["is_valid"]:
        print(
            "Omega/Omega_c check not valid for inhomogeneous run: "
            f"Omega={inhomogeneous_ratio_check['omega']}, Omega_c={inhomogeneous_ratio_check['omega_c']}, "
            f"smallest Nj={inhomogeneous_ratio_check['min_nj']}, ratio={inhomogeneous_ratio_check['ratio']}"
        )
        return None

    # t0 = time.perf_counter()
    # homogeneous_ensemble = run_trajectory_ensemble(
    #     N=N,
    #     Gamma=Gamma,
    #     phases=phases,
    #     sector_coeffs=homogeneous_sector_coeffs,
    #     dt=dt,
    #     num_snapshots=num_snapshots,
    #     seed=seed,
    #     ntraj=ntraj,
    #     shifted_jump_operator=shifted_jump_operator,
    #     n_processes=n_processes,
    #     chunksize=1,
    #     verbose=True,
    # )
    # homogeneous_runtime = time.perf_counter() - t0
    # homogeneous_observables = ensemble_observables(
    #     homogeneous_ensemble,
    #     n_processes=n_processes,
    # )
    # homogeneous_avg = make_averaged_result(homogeneous_ensemble, homogeneous_observables)

    t0 = time.perf_counter()
    inhomogeneous_ensemble = run_trajectory_ensemble(
        N=N,
        Gamma=Gamma,
        phases=phases,
        sector_coeffs=inhomogeneous_sector_coeffs,
        dt=dt,
        num_snapshots=num_snapshots,
        seed=seed,
        ntraj=ntraj,
        shifted_jump_operator=shifted_jump_operator,
        omega_1=omega_1,
        N1=N1,
        N2=N2,
        n_processes=n_processes,
        chunksize=1,
        verbose=True,
    )
    # inhomogeneous_runtime = time.perf_counter() - t0
    inhomogeneous_observables = ensemble_observables(
        inhomogeneous_ensemble,
        n_processes=n_processes,
    )
    inhomogeneous_avg = make_averaged_result(inhomogeneous_ensemble, inhomogeneous_observables)

    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    


compare_homogeneous_vs_inhomogeneous(
    N=10,
    dN=0,
    N1=5,
    dt=1e-2,
    Gamma=1.0,
    ntraj=90,
    shifted_jump_operator=True,
    omega_1=0.7,
    n_processes=9,
)
